# AlphaGenome track matching

Compares the pretrained AlphaGenome RNA and ATAC output tracks with the cardioid
multiome tracks and selects the best-matching AlphaGenome tracks per cell type. The
printed CURIE lists are the track aggregations used for pretrained-AlphaGenome VEP
(`new_track_aggregations.yaml`: CM and early CM, pluripotent and naive heart tracks).

**Inputs** (in `track_matching_dir`): the outputs of the track-matching scripts,
`RNA/AG_Multiome_RNA_feature_sum_exon_counts_corr_500KB.h5ad` (exon counts per gene) and
`ATAC/AG_Multiome_ATAC_stats_enhancer_regions_500KB.parquet` (base-resolution correlations
per 500 kb interval). Track metadata is fetched with the AlphaGenome API
(key from `ALPHAGENOME_API_KEY` or `.env`).

In [ ]:
#Notebook to vizulize track matching:
import numpy as np
import pandas as pd
import seaborn as sns
import anndata as ad
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from adjustText import adjust_text

from cardiomics_s2f.config import get_alphagenome_api_key, load_paths
from cardiomics_s2f.eval.track_matching import (
    extreme_correlation_pairs,
    plot_atac_rna_track_overlap,
    plot_atac_track_match_scatter,
    plot_kde_density_scatter,
    plot_rna_track_match_scatter,
    plot_track_match_correlations_by_assay,
    plot_track_match_scatter_by_assay,
    print_top_matching_tracks,
    top_matching_tracks_per_target,
)

# Ensure plots render inline if you are using Jupyter
%matplotlib inline

# Set general plotting style
sns.set_theme(style="whitegrid")

In [ ]:
from alphagenome.models import dna_client

model = dna_client.create(get_alphagenome_api_key())

atac_metadata = model.output_metadata(
    organism=dna_client.Organism.HOMO_SAPIENS
).atac

rna_metadata = model.output_metadata(
    organism=dna_client.Organism.HOMO_SAPIENS
).rna_seq

In [ ]:
#load track matching data

BASE_DIR = load_paths().track_matching_dir
RNA_adata = ad.read_h5ad(BASE_DIR / "RNA/AG_Multiome_RNA_feature_sum_exon_counts_corr_500KB.h5ad")
ATAC_data = pd.read_parquet(BASE_DIR / "ATAC/AG_Multiome_ATAC_stats_enhancer_regions_500KB.parquet")

# RNA

In [ ]:
#preprocess RNA data
is_pred = RNA_adata.var["data_source"] == "AG"
is_actual = RNA_adata.var["data_source"] == "multiome"

pred_names = RNA_adata.var_names[is_pred]
actual_names = RNA_adata.var_names[is_actual]

RNA_adata.var["index"] = RNA_adata.var.loc[is_pred, ["biosample_name", "Assay title", "strand_type"]].agg("_".join, axis=1)
rna_pearson_matrix = RNA_adata.varm["pear_r_log_norm_across_genes"][is_pred][:, is_actual]

df_rna_pearson = pd.DataFrame(np.nan_to_num(rna_pearson_matrix, 0), index=pred_names, columns=actual_names)

In [ ]:
#plot RNA cor heatmap
matrix_to_plot_clean = df_rna_pearson.T.values
df_to_plot = pd.DataFrame(matrix_to_plot_clean, index=actual_names, columns=pred_names)
metadata_cols = ["biosample_type", "biosample_life_stage", "Assay title", "strand_type"]
pred_meta = RNA_adata.var.loc[is_pred, metadata_cols]

color_df = pd.DataFrame(index=pred_meta.index)
legend_handles = []
palettes = ["husl", "Set2", "tab10", "Pastel1"]

for i, feature in enumerate(metadata_cols):
    series_obj = pred_meta[feature].astype(object)
    unique_vals = series_obj.dropna().unique()  
    palette = sns.color_palette(palettes[i % len(palettes)], len(unique_vals)).as_hex()
    lut = dict(zip(unique_vals, palette))
    color_df[feature] = series_obj.map(lut).fillna("white")
    
    # Add items to legend
    for val, color in lut.items():
        legend_handles.append(mpatches.Patch(color=color, label=f"{feature}: {val}"))
g = sns.clustermap(
    df_to_plot,                 
    xticklabels=False,         
    yticklabels=True,          
    cmap="icefire", 
    center=0,
    vmin=-1, vmax=1,   
    dendrogram_ratio=0.1,
    figsize=(18, 6),
    cbar_pos=(1.05, 0.05, 0.02, 0.4),
    cbar_kws={'label': 'Pearson Correlation'},
    method='ward',
    col_colors=color_df        
)

g.fig.suptitle("RNA Normalized Gene Correlation: True Multiome vs. AG Predictions", y=1.02, fontsize=16)
g.ax_heatmap.set_xlabel("AlphaGenome Predicted Tracks (Clustered)")
g.ax_heatmap.set_ylabel("True Multiome Labels (Clustered)")

g.figure.legend(
    handles=legend_handles, 
    title='Track Metadata',
    loc='upper left', 
    bbox_to_anchor=(1.05, 0.95), 
    ncol=2,                      
    frameon=False                
)
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# 1. Find the top 5 AlphaGenome tracks for each true multiome track
top_preds_list = []
for col in df_rna_pearson.columns:
    # Grab the top 5 indices (curies) for this specific column
    top_5_for_col = df_rna_pearson[col].nlargest(5).index
    top_preds_list.extend(top_5_for_col)

# Get unique curies (removes duplicates if a pred is top 5 for multiple actuals)
top_preds = pd.Index(top_preds_list).unique()

# 2. Subset the data and transpose it for plotting
# Now df_to_plot has actuals as rows and only our selected preds as columns
df_to_plot = df_rna_pearson.T[top_preds]

# 3. Subset the metadata to match the new columns
metadata_cols = ["biosample_type", "biosample_life_stage", "Assay title", "strand_type"]
pred_meta = RNA_adata.var.loc[top_preds, metadata_cols]
#pred_meta.set_index("index", inplace=True)

# 4. Map the colors for the subset
color_df = pd.DataFrame(index=pred_meta.index)
legend_handles = []
palettes = ["husl", "Set2", "tab10", "Pastel1"]

for i, feature in enumerate(metadata_cols):
    series_obj = pred_meta[feature].astype(object)
    unique_vals = series_obj.dropna().unique()  
    palette = sns.color_palette(palettes[i % len(palettes)], len(unique_vals)).as_hex()
    lut = dict(zip(unique_vals, palette))
    color_df[feature] = series_obj.map(lut).fillna("white")
    
    # Add items to legend
    for val, color in lut.items():
        legend_handles.append(mpatches.Patch(color=color, label=f"{feature}: {val}"))

# 5. Map Curies to Biosample Names for the X-axis labels
curie_to_name = RNA_adata.var['biosample_name'].to_dict()
xtick_labels = [curie_to_name[curie] for curie in top_preds]

# 6. Generate the Clustermap
g = sns.clustermap(
    df_to_plot,                 
    xticklabels=xtick_labels,  # Pass the translated biosample names here!
    yticklabels=True,          
    cmap="rocket", 
    #center=0
    vmin=0, vmax=1,            
    figsize=(20, 10),
    cbar_kws={'label': 'Pearson Correlation'},
    method='ward',
    col_colors=color_df        
)

# 7. Rotate the new readable x-axis labels so they don't overlap
plt.setp(g.ax_heatmap.get_xticklabels(), rotation=45, ha='right', fontsize=9)

# Updated title to reflect the new subsetting logic
g.fig.suptitle("RNA Pearson Correlation: True Multiome vs. Top 5 AG Predictions per Actual", y=1.02, fontsize=16)
g.ax_heatmap.set_xlabel("AlphaGenome Predicted Tracks (Clustered)")
g.ax_heatmap.set_ylabel("True Multiome Labels (Clustered)")

g.figure.legend(
    handles=legend_handles, 
    title='Track Metadata',
    loc='upper left', 
    bbox_to_anchor=(1.05, 0.95), 
    ncol=2,                      
    frameon=False                
)

plt.show()

In [ ]:
# --- Define your subset of cell types to plot here ---
melted_rna = df_rna_pearson.T.unstack().reset_index()
melted_rna.columns = ['AG_Track', 'Target_Track', 'Correlation']

curie_to_name = RNA_adata.var['biosample_name'].to_dict()
melted_rna['AG_Biosample_Name'] = melted_rna['AG_Track'].map(curie_to_name)

plt.figure(figsize=(16, 7)) # Slightly narrower since we are plotting a subset
ax = sns.boxplot(data=melted_rna, x='Target_Track', y='Correlation', color="orange", showfliers=False, width = 0.5)
sns.stripplot(data=melted_rna, x='Target_Track', y='Correlation', color="black", alpha=1, size=2, jitter=True, ax=ax)

plt.xticks(rotation=45, ha='right')
plt.title("RNA Pearson Correlation for Selected Cell Types", fontsize=16)

# Annotate Top 5 using biosample_name
texts = []
groups = melted_rna.groupby('Target_Track', sort=False)
x_ticks = [t.get_text() for t in ax.get_xticklabels()]

for group_name, group_df in groups:
    if group_name in x_ticks:
        x_pos = x_ticks.index(group_name)
        top_5 = group_df.nlargest(5, 'Correlation')
        for _, row in top_5.iterrows():
            # Use the newly mapped readable name instead of the AG_Track index
            readable_name = str(row['AG_Biosample_Name'])
            texts.append(ax.text(x_pos, row['Correlation'], readable_name, fontsize=9, color='darkred'))

# Push labels apart to prevent overlap
adjust_text(texts, arrowprops=dict(arrowstyle="-", color='gray', lw=0.5), ax=ax)
plt.show()

In [ ]:
pairs_to_plot = extreme_correlation_pairs(df_rna_pearson)

fig, axes = plt.subplots(1, 4, figsize=(24, 6))
fig.suptitle("RNA Log1p Gene Norm Counts: Predicted vs Observed (Extremes)", fontsize=16)

for ax, pair in zip(axes, pairs_to_plot):
    target = pair['Target_Track']
    ag_pred = pair['AG_Track']
    r_val = pair['Correlation']
    
    # Extract data from AnnData X matrix (log1p transformed)
    obs_vals = RNA_adata[:, target].layers["log_norm_counts"].flatten()
    pred_vals = RNA_adata[:, ag_pred].layers["log_norm_counts"].flatten()
    
    title = f"Target: {target}\nPred: {ag_pred}\nPearson R: {r_val:.3f}"
    plot_kde_density_scatter(obs_vals, pred_vals, ax, title)

plt.tight_layout()
plt.show()

In [ ]:
# ATAC

In [ ]:
ATAC_data

In [ ]:
mean_atac_r = ATAC_data.groupby(["alphagenome_track", "BigWig_track"]).agg(
    r=("pearson_r", "median"),
).reset_index()

df_atac_pearson = mean_atac_r.pivot(index="BigWig_track", columns="alphagenome_track", values="r")

In [ ]:
# 1. Use ALL AlphaGenome tracks
# Transpose so BigWigs are rows and AG tracks are columns
df_to_plot_atac = df_atac_pearson

# 2. Prepare the Metadata
# Set biosample_name as the index so it perfectly aligns with our correlation matrix indices
atac_meta_indexed = atac_metadata.drop_duplicates(subset='biosample_name').set_index('biosample_name')

# Choose the metadata columns you want to visualize as color ribbons
metadata_cols_atac = ["biosample_type", "biosample_life_stage"] 

# Extract metadata for all AG tracks (the index of df_atac_pearson before transposing)
pred_meta_atac = atac_meta_indexed.loc[df_atac_pearson.columns, metadata_cols_atac]

# 3. Map the colors for the metadata
color_df_atac = pd.DataFrame(index=pred_meta_atac.index)
legend_handles_atac = []
palettes = ["husl", "Set2", "tab10", "Pastel1"]

for i, feature in enumerate(metadata_cols_atac):
    series_obj = pred_meta_atac[feature].astype(object)
    unique_vals = series_obj.dropna().unique()  
    palette = sns.color_palette(palettes[i % len(palettes)], len(unique_vals)).as_hex()
    lut = dict(zip(unique_vals, palette))
    color_df_atac[feature] = series_obj.map(lut).fillna("white")
    
    # Add items to legend
    for val, color in lut.items():
        legend_handles_atac.append(mpatches.Patch(color=color, label=f"{feature}: {val}"))

# 4. Generate the Clustermap
g = sns.clustermap(
    df_to_plot_atac,                 
    xticklabels=False, # Hidden to prevent overcrowding         
    yticklabels=True,          
    cmap="rocket", 
    #center = 0, 
    vmin=0, vmax=1,           
    figsize=(18, 8),
    cbar_kws={'label': 'Median Pearson Correlation'},
    method='ward',
    col_colors=color_df_atac        
)

# 5. Add Titles and Labels
g.fig.suptitle("ATAC Pearson Correlation: True Multiome vs. All AG Predictions", y=1.02, fontsize=16)
g.ax_heatmap.set_xlabel("All AlphaGenome Predicted Tracks (Clustered)")
g.ax_heatmap.set_ylabel("True Multiome BigWigs (Clustered)")

# 6. Add the Metadata Legend
g.figure.legend(
    handles=legend_handles_atac, 
    title='Track Metadata',
    loc='upper left', 
    bbox_to_anchor=(1.05, 0.95), 
    ncol=1,                      
    frameon=False                
)

plt.show()

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# 1. Find the top 5 AlphaGenome tracks for each true multiome BigWig
top_preds_list_atac = []

# Iterate over each row (true BigWig) in the dataframe
for row in df_atac_pearson.index:
    # Grab the top 5 columns (AG predictions) for this specific row
    top_5_for_row = df_atac_pearson.loc[row].nlargest(5).index
    top_preds_list_atac.extend(top_5_for_row)

# Get unique predictions (removes duplicates if an AG track is top 5 for multiple BigWigs)
top_preds_atac = pd.Index(top_preds_list_atac).unique()

# Subset the correlation matrix 
# BigWigs remain as rows and our selected unique AG tracks are columns
df_to_plot_atac = df_atac_pearson[top_preds_atac]

# 2. Prepare the Metadata
# Set biosample_name as the index so it perfectly aligns with our correlation matrix indices
# We drop duplicates just in case there are multiple entries for the same biosample name
atac_meta_indexed = atac_metadata.drop_duplicates(subset='biosample_name').set_index('biosample_name')

# Choose the metadata columns you want to visualize as color ribbons
metadata_cols_atac = ["biosample_type", "biosample_life_stage"] 
# Subset metadata using our new dynamic list of top predictions
pred_meta_atac = atac_meta_indexed.loc[top_preds_atac, metadata_cols_atac]

# 3. Map the colors for the metadata
color_df_atac = pd.DataFrame(index=pred_meta_atac.index)
legend_handles_atac = []
palettes = ["husl", "Set2", "tab10", "Pastel1"]

for i, feature in enumerate(metadata_cols_atac):
    series_obj = pred_meta_atac[feature].astype(object)
    unique_vals = series_obj.dropna().unique()  
    palette = sns.color_palette(palettes[i % len(palettes)], len(unique_vals)).as_hex()
    lut = dict(zip(unique_vals, palette))
    color_df_atac[feature] = series_obj.map(lut).fillna("white")
    
    # Add items to legend
    for val, color in lut.items():
        legend_handles_atac.append(mpatches.Patch(color=color, label=f"{feature}: {val}"))

# 4. Generate the Clustermap
g = sns.clustermap(
    df_to_plot_atac,                 
    xticklabels=True,  # Set to True so the biosample_names are displayed         
    yticklabels=True,          
    cmap="rocket", 
    vmin=0, vmax=1,            
    figsize=(18, 10),  # Made slightly taller to accommodate the legend and x-labels
    cbar_kws={'label': 'Median Pearson Correlation'},
    method='ward',
    col_colors=color_df_atac        
)

# 5. Rotate x-axis labels to prevent overlapping
plt.setp(g.ax_heatmap.get_xticklabels(), rotation=45, ha='right', fontsize=9)

# 6. Add Titles and Labels
# Updated title to reflect the per-row logic
g.fig.suptitle("ATAC Pearson Correlation: True Multiome vs. Top 5 AG Predictions per BigWig", y=1.02, fontsize=16)
g.ax_heatmap.set_xlabel("AlphaGenome Predicted Tracks (Clustered)")
g.ax_heatmap.set_ylabel("True Multiome BigWigs (Clustered)")

# 7. Add the Metadata Legend
g.figure.legend(
    handles=legend_handles_atac, 
    title='Track Metadata',
    loc='upper left', 
    bbox_to_anchor=(1.05, 0.95), 
    ncol=1,                      
    frameon=False                
)

plt.show()

In [ ]:
#plot ATAC boxplots
melted_atac = df_atac_pearson.T.unstack().reset_index()
melted_atac.columns = ['Target_Track', 'AG_Track', 'Correlation']

plt.figure(figsize=(18, 5))
ax = sns.boxplot(data=melted_atac, x='Target_Track', y='Correlation', color="lightblue", showfliers=False, width = 0.5)
sns.stripplot(data=melted_atac, x='Target_Track', y='Correlation', color="black", alpha=1, size=2, jitter=True, ax=ax)

plt.xticks(rotation=45, ha='right')
plt.title("ATAC Pearson Correlation Distributions per Cell Type & Day", fontsize=16)

texts = []
groups = melted_atac.groupby('Target_Track', sort=False)
x_ticks = [t.get_text() for t in ax.get_xticklabels()]

for group_name, group_df in groups:
    if group_name in x_ticks:
        x_pos = x_ticks.index(group_name)
        top_5 = group_df.nlargest(5, 'Correlation')
        for _, row in top_5.iterrows():
            texts.append(ax.text(x_pos, row['Correlation'], row['AG_Track'], fontsize=7, color='darkblue'))

adjust_text(texts, arrowprops=dict(arrowstyle="-", color='gray', lw=0.5), ax=ax)
plt.show()

In [ ]:
# Execution
target_types = ['Cardiomyocytes',  'Fibroblasts', 'Cardiac-progenitors', 'Cardiac-mesoderm',]

plot_track_match_correlations_by_assay(
    df_atac=df_atac_pearson, 
    df_rna=df_rna_pearson, 
    rna_adata=RNA_adata, 
    target_cell_types=target_types
)
plt.show()

In [ ]:
#atac scatter plots
atac_pairs = extreme_correlation_pairs(df_atac_pearson)

fig, axes = plt.subplots(1, 4, figsize=(24, 6))
fig.suptitle("ATAC Log1p Counts: Predicted vs Observed Full Interval (Extremes)", fontsize=16)

for ax, pair in zip(axes, atac_pairs):
    ag_pred = pair['Target_Track']
    target = pair['AG_Track']
    r_val = pair['Correlation']
    
    # Extract interval sums from our CSVs and log1p transform them
    obs_vals = ATAC_data.loc[(ATAC_data.BigWig_track == target) & (ATAC_data.alphagenome_track == ag_pred), "log_sum_obs_count"].values
    pred_vals = ATAC_data.loc[(ATAC_data.BigWig_track == target) & (ATAC_data.alphagenome_track == ag_pred), "log_sum_pred_count"].values
        
    title = f"Target: {target}\nPred: {ag_pred}\nbp res. Pearson R: {r_val:.3f}"
    plot_kde_density_scatter(obs_vals, pred_vals, ax, title)

plt.tight_layout()
plt.show()

# Select aggregation tracks

In [ ]:
atac_consensus_tracks = top_matching_tracks_per_target(df_atac_pearson, top_n=10)
print_top_matching_tracks(atac_consensus_tracks)

In [ ]:
df_rna_pearson_reidx = df_rna_pearson.T.copy() 
df_rna_pearson_reidx.columns = RNA_adata.var.loc[is_pred, ["biosample_name", "Assay title", "strand_type"]].agg("_".join, axis=1)
rna_consensus_tracks = top_matching_tracks_per_target(df_rna_pearson_reidx, top_n=10)
print_top_matching_tracks(rna_consensus_tracks)

In [ ]:
metadata = model.output_metadata(
    dna_client.Organism.HOMO_SAPIENS
).concatenate()

## Select CM tracks

In [ ]:
#check biosample type for number of tracks with differnt modalities
CM_atac_tracks = [track[0] for track in atac_consensus_tracks["Cardiomyocytes"] ] #+ atac_consensus_tracks["Early_cardiomyocytes"]
CM_rna_tracks = [track[0].split("_")[0] for track in rna_consensus_tracks["Cardiomyocytes"] + rna_consensus_tracks["Early-cardiomyocytes"]]
CM_tracks = {*CM_atac_tracks} & {*CM_rna_tracks}

CM_metadata = metadata[metadata.biosample_name.isin(CM_tracks)]
CM_metadata.groupby("biosample_name").size().sort_values(ascending=False)

In [ ]:
# Execution using your defined sets
plot_atac_rna_track_overlap(
    atac_set=CM_atac_tracks, 
    rna_set=CM_rna_tracks, 
    title="Cardiomyocytes",
    figsize=(4, 3)
)

In [ ]:
print("(Early) Cardiomyocytes:")
for __, row in  CM_metadata[["ontology_curie", "biosample_name"]].drop_duplicates().iterrows():
    print(f'  - "{row["ontology_curie"]}"  #{row["biosample_name"]}')

In [ ]:
tracks = [track for track in CM_tracks if track in atac_metadata.biosample_name.to_list()]

plot_atac_track_match_scatter(ATAC_data, df_atac_pearson, ag_tracks = tracks, bw_target="Early_cardiomyocytes", n_rows = 1)

In [ ]:
tracks = [track for track in CM_tracks if track in atac_metadata.biosample_name.to_list()]
plot_atac_track_match_scatter(ATAC_data, df_atac_pearson, ag_tracks = tracks, bw_target="Cardiomyocytes", n_rows = 1)

In [ ]:
tracks = [track for track in CM_tracks if track in rna_metadata.biosample_name.to_list()]
plot_rna_track_match_scatter(RNA_adata, df_rna_pearson, ag_tracks = tracks, bw_target="Early-cardiomyocytes", n_rows = 1)

In [ ]:
tracks = [track for track in CM_tracks if track in rna_metadata.biosample_name.to_list()]
plot_rna_track_match_scatter(RNA_adata, df_rna_pearson, ag_tracks = tracks, bw_target="Cardiomyocytes", n_rows = 1)

# Progenitor tracks

In [ ]:
EE_rna_tracks = {track[0].split("_")[0] for track in rna_consensus_tracks["Early-endoderm"]}
CME_rna_tracks = {track[0].split("_")[0] for track in rna_consensus_tracks["Cardiac-mesoderm"]}
CP_rna_tracks = {track[0].split("_")[0] for track in rna_consensus_tracks["Cardiac-progenitors"]}
ED_rna_tracks = {track[0].split("_")[0] for track in rna_consensus_tracks["Ectoderm"]}

from collections import Counter
progenitor_tracks = list(EE_rna_tracks) + list(CME_rna_tracks) + list(CP_rna_tracks) + list(ED_rna_tracks)
progenitor_counts = Counter(progenitor_tracks)
progenitor_tracks = pd.DataFrame(
    [(track, count) for track, count in progenitor_counts.items() if count >= 2],
    columns=["Track", "Total_Occurrences"]
)
progenitor_tracks

In [ ]:
#from this it appears like we have a stemm cell like population. The problem is that the H1, H7 and H9 line are not in ATAC data so we need to pick simialr ones for those as well.
#define pluipotnetn lines from this set manually
progenitor_tracks = ["H1", "WTC11", "GM23338", "H9", "HUES64", "H7"]
progenitor_metadata = metadata[metadata.biosample_name.isin(progenitor_tracks)]
progenitor_metadata.groupby("biosample_name").size().sort_values(ascending=False)

In [ ]:
print("Progenitors:")
for __, row in  progenitor_metadata[["ontology_curie", "biosample_name"]].drop_duplicates().iterrows():
    print(f'  - "{row["ontology_curie"]}"  #{row["biosample_name"]}')

In [ ]:
progenitor_metadata["Assay title"].unique()

In [ ]:
tracks = [track for track in progenitor_tracks if track in atac_metadata.biosample_name.to_list()]
plot_atac_track_match_scatter(ATAC_data, df_atac_pearson, ag_tracks = tracks, bw_target="Cardiac_progenitors", n_rows = 1)

In [ ]:
tracks = [track for track in progenitor_tracks if track in rna_metadata.biosample_name.to_list()]
plot_rna_track_match_scatter(RNA_adata, df_rna_pearson, ag_tracks = tracks, bw_target="Cardiac-progenitors", n_rows = 1)

## get naive aggregation

In [ ]:
naive_tracks = metadata.loc[metadata.biosample_name.str.contains(r"heart|cardi", case=False, na=False),
             ["ontology_curie", "biosample_name"]].drop_duplicates()

for __, row in naive_tracks.iterrows():
    print(f'  - "{row["ontology_curie"]}"  #{row["biosample_name"]}')

In [ ]:
tracks = [track for track in naive_tracks.biosample_name if track in atac_metadata.biosample_name.to_list()]
plot_atac_track_match_scatter(ATAC_data, df_atac_pearson, ag_tracks = tracks, bw_target="Cardiomyocytes", n_rows = 2)

In [ ]:
tracks = [track for track in naive_tracks.biosample_name if track in rna_metadata.biosample_name.to_list()]
plot_rna_track_match_scatter(RNA_adata, df_rna_pearson, ag_tracks = tracks, bw_target="Cardiomyocytes", n_rows = 2)

In [ ]:
# --- Execution ---

atac_tracks_subset = [track for track in naive_tracks.biosample_name if track in atac_metadata.biosample_name.to_list()]
rna_tracks_subset = [track for track in naive_tracks.biosample_name if track in rna_metadata.biosample_name.to_list()]

plot_track_match_scatter_by_assay(
    atac_data=ATAC_data,
    df_atac_pearson=df_atac_pearson,
    atac_tracks=atac_tracks_subset,
    rna_adata=RNA_adata,
    df_rna_pearson=df_rna_pearson,
    rna_tracks=rna_tracks_subset,
    bw_target="Cardiomyocytes",
    n_cols=8
)